# NYC Taxi Data Cleaning
Loads 4 raw CSV files, aligns columns, cleans, and saves a valid dataset.

In [ ]:
import pandas as pd
import os

RAW_FILES = [
    '../Data/raw/yellow_tripdata_2015-01.csv',
    '../Data/raw/yellow_tripdata_2016-01.csv',
    '../Data/raw/yellow_tripdata_2016-02.csv',
    '../Data/raw/yellow_tripdata_2016-03.csv',
]
OUTPUT_PATH = '../Data/cleaned/cleaned_taxi_data.csv'

In [ ]:
# Canonical column name mapping — normalise before concat
COLUMN_RENAMES = {
    'RateCodeID': 'RatecodeID',   # 2015 uses RateCodeID, 2016 uses RatecodeID
    'rate_code':  'RatecodeID',
}

frames = []
for path in RAW_FILES:
    tmp = pd.read_csv(path, low_memory=False)
    # Strip whitespace from column names (common hidden issue)
    tmp.columns = tmp.columns.str.strip()
    # Normalise known column name variants
    tmp.rename(columns=COLUMN_RENAMES, inplace=True)
    print(f'Loaded {os.path.basename(path)}: {tmp.shape}')
    frames.append(tmp)

# Align all frames to a common column set before concat
all_cols = sorted(set(col for f in frames for col in f.columns))
frames = [f.reindex(columns=all_cols) for f in frames]

df = pd.concat(frames, ignore_index=True)
print('Step 1 (after concat):', df.shape)

In [ ]:
df['tpep_pickup_datetime']  = pd.to_datetime(df['tpep_pickup_datetime'],  errors='coerce')
df['tpep_dropoff_datetime'] = pd.to_datetime(df['tpep_dropoff_datetime'], errors='coerce')

nat_count = df['tpep_pickup_datetime'].isna().sum() + df['tpep_dropoff_datetime'].isna().sum()
print(f'Step 2 (datetime conversion): {df.shape} | NaT values: {nat_count}')

In [ ]:
df['trip_duration'] = (
    df['tpep_dropoff_datetime'] - df['tpep_pickup_datetime']
).dt.total_seconds() / 60  # minutes

null_duration = df['trip_duration'].isna().sum()
print(f'Step 3 (trip_duration computed): {df.shape} | NaN durations: {null_duration}')

In [ ]:
CRITICAL_COLS = [
    'tpep_pickup_datetime',
    'tpep_dropoff_datetime',
    'trip_distance',
    'fare_amount',
    'trip_duration',
]

before = df.shape[0]
df.dropna(subset=CRITICAL_COLS, inplace=True)
print(f'Step 4 (selective dropna): {df.shape} | Rows dropped: {before - df.shape[0]}')

In [ ]:
before = df.shape[0]

mask = (
    (df['trip_distance'] > 0)    & (df['trip_distance'] < 50)   &
    (df['fare_amount']   > 0)    & (df['fare_amount']   < 500)  &
    (df['trip_duration'] > 0)    & (df['trip_duration'] < 120)
)

df = df[mask]
print(f'Step 5 (filtering): {df.shape} | Rows removed: {before - df.shape[0]}')

if df.shape[0] == 0:
    print('WARNING: All rows were filtered out. Check your filter bounds or upstream data.')

In [ ]:
df['pickup_hour']    = df['tpep_pickup_datetime'].dt.hour
df['pickup_weekday'] = df['tpep_pickup_datetime'].dt.dayofweek
df['pickup_month']   = df['tpep_pickup_datetime'].dt.month

print(f'Step 6 (feature engineering): {df.shape}')

In [ ]:
if df.shape[0] == 0:
    print('ERROR: DataFrame is empty. Skipping save.')
else:
    os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)
    df.to_csv(OUTPUT_PATH, index=False)
    print(f'Saved {df.shape[0]:,} rows to {OUTPUT_PATH}')